In [1]:
print("OK")

OK


In [2]:
import pandas as pd
import numpy as np

In [7]:
books = pd.read_csv('BX-Books.csv', sep=';', on_bad_lines='skip', encoding='latin-1')

C:\Users\amosi\AppData\Local\Temp\ipykernel_80868\800602712.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('BX-Books.csv', sep=';', on_bad_lines='skip', encoding='latin-1')


In [8]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [9]:
books = books.rename(columns={'Book-Title': 'title', 'Book-Author': 'author', 'Year-Of-Publication': 'year', 'Publisher': 'publisher', 'Image-URL-L': 'image_url'}) 

In [11]:
books = books[['ISBN','title', 'author', 'year', 'publisher', 'image_url']]    

In [12]:
users = pd.read_csv('BX-Users.csv', sep=';', on_bad_lines='skip', encoding='latin-1')

In [13]:
users.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [14]:
ratings = pd.read_csv('BX-Book-Ratings.csv', sep=';', on_bad_lines='skip', encoding='latin-1')

In [15]:
ratings.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [16]:
ratings.shape

(1149780, 3)

In [17]:
ratings.rename(columns={'Book-Rating': 'rating'}, inplace=True)

In [18]:
ratings.head()

,User-ID,ISBN,rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [19]:
print('books shape:', books.shape, 'users shape:', users.shape, 'ratings shape:', ratings.shape, sep='\n')

books shape:
(271360, 6)
users shape:
(278858, 3)
ratings shape:
(1149780, 3)


In [21]:
ratings['User-ID'].value_counts()

User-ID
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
          ...  
119573        1
276706        1
276697        1
276679        1
276676        1
Name: count, Length: 105283, dtype: int64

In [ ]:
ratings_per_user = ratings['User-ID'].value_counts()
cutoff = ratings_per_user.quantile(0.90)  # top 10% most active users
print(cutoff)


12.0


In [ ]:
ratings_per_user.head()

User-ID
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
Name: count, dtype: int64

In [25]:
x = ratings['User-ID'].value_counts() > 200

In [27]:
x

User-ID
11676      True
198711     True
153662     True
98391      True
35859      True
          ...  
119573    False
276706    False
276697    False
276679    False
276676    False
Name: count, Length: 105283, dtype: bool

In [28]:
y = x[x].index

In [29]:
ratings = ratings[ratings['User-ID'].isin(y)]

In [30]:
ratings.head()

,User-ID,ISBN,rating
1456,277427,002542730X,10
1457,277427,0026217457,0
1458,277427,003008685X,8
1459,277427,0030615321,0
1460,277427,0060002050,0


In [31]:
ratings_with_books = ratings.merge(books, on='ISBN')

In [32]:
number_ratings = ratings_with_books.groupby('title')['rating'].count().reset_index()

In [33]:
number_ratings.head()   
number_ratings.rename(columns={'rating': 'number_of_ratings'}, inplace=True)    

In [34]:
number_ratings.head()

,title,number_of_ratings
0,A Light in the Storm: The Civil War Diary of ...,2
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,Beyond IBM: Leadership Marketing and Finance ...,1
4,Clifford Visita El Hospital (Clifford El Gran...,1


In [35]:
final_ratings = ratings_with_books.merge(number_ratings, on='title')

In [36]:
final_ratings = final_ratings[final_ratings['number_of_ratings'] >= 50]

In [37]:
final_ratings.shape

(61853, 9)

In [38]:
final_ratings.drop_duplicates(['User-ID', 'title'], inplace=True)

In [47]:
book_pivot = final_ratings.pivot_table(columns='User-ID', index='title', values='rating')

In [48]:
book_pivot.fillna(0, inplace=True)

In [49]:
from scipy.sparse import csr_matrix
book_sparse = csr_matrix(book_pivot)

In [50]:
book_sparse

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 14961 stored elements and shape (742, 888)>

In [51]:
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(algorithm='brute')

In [52]:
model.fit(book_sparse)

,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'minkowski'
,p,2
,metric_params,None
,n_jobs,None


In [53]:
book_pivot.iloc[273,:]

User-ID
254       0.0
2276      0.0
2766      0.0
2977      0.0
3363      0.0
         ... 
275970    0.0
277427    8.0
277478    0.0
277639    0.0
278418    0.0
Name: If Only It Were True, Length: 888, dtype: float64

In [54]:
distance, suggestion = model.kneighbors(book_pivot.iloc[273,:].values.reshape(1,-1), n_neighbors=6) 

In [55]:
distance

array([[ 0.        , 26.41968963, 27.47726333, 27.80287755, 27.85677655,
        28.31960452]])

In [56]:
suggestion

array([[273, 372, 604, 393, 536, 320]])

In [57]:
for i in range(len(suggestion)):
    print(book_pivot.index[suggestion[i]])

Index(['If Only It Were True', 'No Safe Place', 'The Most Wanted',
       'Pleading Guilty', 'The Cradle Will Fall', 'Long After Midnight'],
      dtype='object', name='title')


In [70]:
book_names = book_pivot.index

In [72]:
book_names

Index(['1984', '1st to Die: A Novel', '2nd Chance', '4 Blondes',
       '84 Charing Cross Road', 'A Bend in the Road', 'A Case of Need',
       'A Child Called \It\": One Child's Courage to Survive"',
       'A Civil Action', 'A Cry In The Night',
       ...
       'Winter Solstice', 'Wish You Well', 'Without Remorse',
       'Wizard and Glass (The Dark Tower, Book 4)', 'Wuthering Heights',
       'Year of Wonders', 'You Belong To Me',
       'Zen and the Art of Motorcycle Maintenance: An Inquiry into Values',
       'Zoya', '\O\" Is for Outlaw"'],
      dtype='object', name='title', length=742)

In [59]:
np.where(book_pivot.index == '1984')[0][0]

np.int64(0)

In [61]:
ids = np.where(final_ratings['title'] == '1984')[0][0]  

In [62]:
final_ratings.iloc[ids]['image_url']

'http://images.amazon.com/images/P/0451524934.01.LZZZZZZZ.jpg'

In [63]:
book_name = []
for book_id in suggestion[0]:
    book_name.append(book_pivot.index[book_id])

In [64]:
book_name

['If Only It Were True',
 'No Safe Place',
 'The Most Wanted',
 'Pleading Guilty',
 'The Cradle Will Fall',
 'Long After Midnight']

In [65]:
ids_index = []
for name in book_name:
    ids_index.append(np.where(final_ratings['title'] == name)[0][0])

In [67]:
ids_index

[np.int64(89),
 np.int64(22),
 np.int64(710),
 np.int64(784),
 np.int64(2297),
 np.int64(76)]

In [68]:
for idx in ids_index:
    url = final_ratings.iloc[idx]['image_url']
    print(url)

http://images.amazon.com/images/P/0743406176.01.LZZZZZZZ.jpg
http://images.amazon.com/images/P/0345404777.01.LZZZZZZZ.jpg
http://images.amazon.com/images/P/0451196856.01.LZZZZZZZ.jpg
http://images.amazon.com/images/P/0446365505.01.LZZZZZZZ.jpg
http://images.amazon.com/images/P/0440115450.01.LZZZZZZZ.jpg
http://images.amazon.com/images/P/0553571818.01.LZZZZZZZ.jpg


In [73]:
book_names

Index(['1984', '1st to Die: A Novel', '2nd Chance', '4 Blondes',
       '84 Charing Cross Road', 'A Bend in the Road', 'A Case of Need',
       'A Child Called \It\": One Child's Courage to Survive"',
       'A Civil Action', 'A Cry In The Night',
       ...
       'Winter Solstice', 'Wish You Well', 'Without Remorse',
       'Wizard and Glass (The Dark Tower, Book 4)', 'Wuthering Heights',
       'Year of Wonders', 'You Belong To Me',
       'Zen and the Art of Motorcycle Maintenance: An Inquiry into Values',
       'Zoya', '\O\" Is for Outlaw"'],
      dtype='object', name='title', length=742)

In [74]:
import pickle
pickle.dump(model, open('artifacts/model.pkl', 'wb'))
pickle.dump(book_names, open('artifacts/book_pivot.pkl', 'wb'))
pickle.dump(final_ratings, open('artifacts/final_ratings.pkl', 'wb'))
pickle.dump(book_pivot, open('artifacts/book_pivot_table.pkl', 'wb'))


TESTING MODEL

In [78]:
def the_recommend_books(book_name):
    book_id = np.where(book_pivot.index == book_name)[0][0]
    distance, suggestion = model.kneighbors(book_pivot.iloc[book_id,:].values.reshape(1,-1), n_neighbors=6) 

    book_name = []
    for book_id in suggestion[0]:
        book_name.append(book_pivot.index[book_id])

    ids_index = []
    for name in book_name:
        ids_index.append(np.where(final_ratings['title'] == name)[0][0])

    urls = []
    for idx in ids_index:
        url = final_ratings.iloc[idx]['image_url']
        urls.append(url)
    
    return urls

In [80]:
def recommended_book(book_name):
    book_id = np.where(book_pivot.index == book_name)[0][0]
    distance, suggestion = model.kneighbors(book_pivot.iloc[book_id,:].values.reshape(1,-1), n_neighbors=6) 

    for i in range(len(suggestion)):
        books = book_pivot.index[suggestion[i]]
        for j in books:
            if j == book_name:
                print(f'You searched for book: {book_name}\n')
                print('Recommended books are:\n')
            else:
                print(j)

In [82]:
book_name = 'Wish You Well'
recommended_book(book_name)

You searched for book: Wish You Well

Recommended books are:

Last Man Standing
Exclusive
Jacob Have I Loved
No Safe Place
The Sands of Time
